In [1]:
# Libraries ----
import os
import re
import sys
import warnings
import numpy as np  # type: ignore
import pandas as pd  # type: ignore

sys.path.append("../modules")
import plot_interactive as pi  # type: ignore
import estimate_hoi_measures as ehm  # type: ignore
import estimate_complexity_measures as ecm  # type: ignore
import estimate_synchronization_analysis as esa  # type: ignore
import estimate_complex_network_analysis as ecna  # type: ignore

# Global options ----
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)

## Global variables

In [2]:
log_path = "../logs"
input_path = "../input_files"
output_path = "../output_files"
input_generation_date = "2025-02-18"
x_bounds = [0, 240]
y_bounds = [0, 135]
t_threshold = 3600

## Load and prepare data of all simulated data

In [12]:
cols = [
    "particles",
    "video",
    "permuted_id",
    "time",
    "position_x",
    "position_y",
    "corrected_orientation",
    "n_x",
    "n_y",
    "n_orientation",
    "norm"
]

df_final = []
path = "../../simulate_hoi/output_files"
for file in os.listdir(path):
    df = pd.read_csv(path + "/" + file, low_memory=False)
    df["n_x"] = df["position_x"] / x_bounds[1] - 0.5  # Normalized position in X-axis
    df["n_y"] = df["position_y"] / y_bounds[1] - 0.5  # Normalized position in Y-axis
    df["n_orientation"] = np.cos(df["corrected_orientation"])  # Normalized Orientation
    df["norm"] = (
        np.power(df["n_x"], 2)
        + np.power(df["n_y"], 2)
        + np.power(df["n_orientation"], 2)
    )
    df_final.append(df[df["time"] <= t_threshold][cols])

df_final = pd.concat(df_final, ignore_index=True)
df_final

,particles,video,permuted_id,time,position_x,position_y,corrected_orientation,n_x,n_y,n_orientation,norm
0,2,2n_2m_0f_simulation,0,0,135.504689,60.106917,0.405276,0.064603,-0.054764,0.918994,0.851722
1,2,2n_2m_0f_simulation,0,3,135.504689,60.106917,0.405276,0.064603,-0.054764,0.918994,0.851722
2,2,2n_2m_0f_simulation,0,6,135.504689,60.106917,0.405276,0.064603,-0.054764,0.918994,0.851722
3,2,2n_2m_0f_simulation,0,9,126.187102,33.173061,0.276404,0.025780,-0.254274,0.962043,0.990846
4,2,2n_2m_0f_simulation,0,12,126.187102,33.173061,0.276404,0.025780,-0.254274,0.962043,0.990846
...,...,...,...,...,...,...,...,...,...,...,...
10804,4,4n_4m_0f_simulation,3,3588,3.063519,1.831905,-2.943539,-0.487235,-0.486430,-0.980451,1.435298
10805,4,4n_4m_0f_simulation,3,3591,13.142552,7.868005,0.600600,-0.445239,-0.441718,0.824997,1.073973
10806,4,4n_4m_0f_simulation,3,3594,-4.816526,-6.838236,2.950487,-0.520069,-0.550654,-0.981795,1.537612
10807,4,4n_4m_0f_simulation,3,3597,7.081626,2.265047,1.471705,-0.470493,-0.483222,0.098929,0.464654


## Plot time series of each individual

In [7]:
pi.interactive_plot(df=df_final, interval_size=600)

# Are there higher-order interactions in 3 and 4 cockroach videos?

- Aim: To classify regimes as redundancy-dominated or synergy-dominated over time.
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.

## Get O-information data between individuals
O-Information is a measure used in information theory to quantify **high-order interactions** in multivariate systems. It distinguishes between redundant and synergistic information sharing across multiple variables.

### **Mathematical Formulation**

For a set of random variables $(X_1, X_2, ..., X_N)$, the O-Information and exogenous information are defined as:

\begin{align}
    \Omega(X_1, X_2, ..., X_N) &= TC(\mathbf{X}_{n}) - DTC(\mathbf{X}_{n}) = \sum_{i=1}^{N} I(X_i; \mathbf{X}_{-i}) - I(\mathbf{X}_1, ..., \mathbf{X}_N) \\
    S(X_1, X_2, ..., X_N) &= TC(\mathbf{X}_{n}) + DTC(\mathbf{X}_{n})
\end{align}

respectively, where:

- $I(X_i; \mathbf{X}_{-i})$ is the mutual information between $X_i$ and the rest.
- $I(\mathbf{X}_1, ..., \mathbf{X}_N)$ is the total multivariate mutual information.
- If $\Omega>0$, the system is redundancy-dominated.
- If $\Omega<0$, the system is synergy-dominated.


In [15]:
window_sizes = [
    [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225],
    [240], [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_oinfo = []
for video in df_final["video"].unique():
    df = df_final[df_final["video"] == video]
    if int(video[0]) >= 3:
        df_aux = ehm.estimate_oinfo_multiple_windows(
            df=df,
            window_sizes=window_sizes,
            log_path=log_path,
            log_filename="log_hoi_sim",
            verbose=1,
            tqdm_bar=True
        )
        df_oinfo.append(df_aux)

df_oinfo = pd.concat(df_oinfo, ignore_index=True)
df_oinfo.to_csv(output_path + "/df_hoi_sim.csv", index=False)
df_oinfo

100%|███████████████████████| 23/23 [06:19<00:00, 16.49s/it]


,video,t_range,size,multiplet,oinfo_distance,oinfo_orientation,sinfo_distance,sinfo_orientation
0,3n_3m_0f_simulation,0 - 30,30,012,-0.223218,0.011935,1.382528,-0.424704
1,3n_3m_0f_simulation,30 - 60,30,012,-0.005695,0.012038,-0.310041,-0.397518
2,3n_3m_0f_simulation,60 - 90,30,012,-0.050611,0.059623,0.338505,0.335066
3,3n_3m_0f_simulation,90 - 120,30,012,0.020334,0.006218,-0.385762,-0.366555
4,3n_3m_0f_simulation,120 - 150,30,012,0.023915,0.012045,-0.283748,-0.147323
...,...,...,...,...,...,...,...,...
4519,4n_4m_0f_simulation,0 - 3600,3600,012,0.000075,0.000051,0.008842,0.001515
4520,4n_4m_0f_simulation,0 - 3600,3600,013,-0.000010,0.000023,0.002238,-0.000212
4521,4n_4m_0f_simulation,0 - 3600,3600,023,-0.000017,0.000036,0.004221,0.001623
4522,4n_4m_0f_simulation,0 - 3600,3600,123,0.000007,0.000058,-0.002159,0.002799


---
# How different is the behavior when sex ratio changes (fixed group size)?

- Aim: To estimate Hurst exponent (asses persistence), Permutation entropy (predictability), Statistical complexity (assess the balance of order and randomness) for each cockroach’s time series (distance, orientation) → persistence vs. randomness.
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.
---

## Get Hurst exponent, permutation entropy and Statistical complexity data between individuals

### **Mathematical Formulation**

#### 1. Hurst Exponent

The Hurst exponent ($H$) is a statistical measure used to evaluate the long-term memory of time series data. It quantifies the tendency of a time series to either:

- Persist in its trend ($H > 0.5$)
- Exhibit a random walk ($H ≈ 0.5$)
- Mean-revert (antipersistence) ($H < 0.5$)

The Hurst exponent is estimated using the rescaled range (R/S) analysis:

\begin{equation}
    H = \lim_{T \to \infty} \frac{\log(R/S)}{\log(T)},
\end{equation}

where $R$ is the range of cumulative deviations from the mean, $S$ is the standard deviation, and $T$ is the time window size.

Also, Multifractal Detrended Fluctuation Analysis (MF-DFA) is an extension of DFA (Detrended Fluctuation Analysis) that measures the **multifractal properties** of a time series by analyzing its scaling behavior at different moments $q$. The fluctuation function is defined as:

\begin{equation}
    F_q(s) = \left( \frac{1}{N_s} \sum_{\nu=1}^{N_s} F^q(\nu, s) \right)^{\frac{1}{q}}
\end{equation}

where $F(\nu, s)$ is the local detrended fluctuation at segment $\nu$, and $s$ is the window size.

#### 2. Permutation Entropy

Permutation Entropy ($PE$) is a nonlinear measure of time series complexity introduced by Bandt & Pompe (2002). It quantifies the randomness in a time series by analyzing the frequency of ordinal patterns of a given length.

Given a time series $X = \{x_1, x_2, ..., x_N\}$ and an embedding dimension $d$, we extract ordinal patterns by ranking the values within sliding windows of size $\tau$. The permutation entropy is then computed as:

\begin{equation}
    H_p = - \sum p(\pi) \log p(\pi)
\end{equation}

where $p(\pi)$ is the probability of each ordinal pattern $\pi$.

#### 3. Statistical complexity

Statistical complexity measures the balance between disorder and structure in a system. It complements entropy by identifying structured patterns within randomness.

A widely used definition is the **Jensen-Shannon complexity**, which combines permutation entropy and disequilibrium:

\begin{equation}
    C_J = H_p \cdot Q_J
\end{equation}

where $Q_J$ is the Jensen-Shannon divergence measuring disequilibrium.

---
## **References**

- Bandt, C., & Pompe, B. (2002). Permutation entropy: A natural complexity measure for time series.
- Rosso, O. A., et al. (2007). Distinguishing noise from chaos.
- Ribeiro, H. V., et al. (2012). Characterizing time series through complexity-entropy curves.
- `ordpy`: A Python library for ordinal pattern analysis.
---

In [16]:
window_sizes = [
    # [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225], [240],
    [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_complexity = []
for video in df_final["video"].unique():
    df = df_final[df_final["video"] == video]
    df_aux = ecm.estimate_multiple_hurst_complexity(
        df=df,
        window_sizes=window_sizes,
        q=2,
        dx=3,
        taux=1,
        log_path=log_path,
        log_filename="log_complexity_sim",
        verbose=1,
        tqdm_bar=True
    )

    df_complexity.append(df_aux)

df_complexity = pd.concat(df_complexity, ignore_index=True)
df_complexity.to_csv(output_path + "/df_complexity_sim.csv", index=False)
df_complexity

100%|███████████████████████| 15/15 [00:04<00:00,  3.46it/s]


,video,t_range,size,permuted_id,H_distance,PE_distance,C_distance,H_orientation,PE_orientation,C_orientation
0,2n_2m_0f_simulation,0 - 120,120,0,0.457244,0.738782,0.234654,0.156989,0.811707,0.159713
1,2n_2m_0f_simulation,0 - 120,120,1,NaN,0.672473,0.251749,NaN,0.691644,0.193531
2,2n_2m_0f_simulation,120 - 240,120,0,0.144402,0.826164,0.154665,0.242127,0.665795,0.247312
3,2n_2m_0f_simulation,120 - 240,120,1,-0.132488,0.722282,0.240180,NaN,0.782844,0.161116
4,2n_2m_0f_simulation,240 - 360,120,0,0.582234,0.619209,0.231841,0.205107,0.783259,0.161925
...,...,...,...,...,...,...,...,...,...,...
1624,4n_4m_0f_simulation,1800 - 3600,1800,3,-0.005254,0.927197,0.070913,0.041985,0.919211,0.066964
1625,4n_4m_0f_simulation,0 - 3600,3600,0,0.493164,0.808747,0.162645,0.411131,0.819001,0.148299
1626,4n_4m_0f_simulation,0 - 3600,3600,1,0.006646,0.909160,0.088302,0.030185,0.967721,0.030507
1627,4n_4m_0f_simulation,0 - 3600,3600,2,0.520033,0.832814,0.141591,0.408042,0.772613,0.185973


---
# How different is the behavior when sex ratio changes (fixed group size)?

- Aim: To examine coordination and coupling using Cross-recurrence quantification analysis (CRQA).
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.
---

## Synchronization in Complex Systems

### **Mathematical Formulation**

#### 1. Recurrence Plot Analysis (RPA)

**Recurrence Plot Analysis (RPA)** is a nonlinear method for studying the dynamics and synchronization of complex systems. It visualizes when a system revisits the same or similar state in its phase space.

Given a trajectory $\vec{x}_i \in \mathbb{R}^d$, the **recurrence matrix** is defined as:

\begin{equation}
    R_{i,j} = \Theta(\varepsilon - \|\vec{x}_i - \vec{x}_j\|),
\end{equation}

where:
- $\varepsilon$ is a threshold,
- $\Theta$ is the Heaviside step function,
- $R_{i,j} = 1$ means state $i$ recurs at time $j$.

The resulting binary matrix can be visualized as a **recurrence plot**.

**Recurrence Quantification Analysis (RQA)** converts the visual features of recurrence plots into quantitative metrics:

| Metric | Description |
|--------|-------------|
| **RR** – Recurrence Rate | Ratio of recurrent points in the plot |
| **DET** – Determinism | Fraction of recurrent points forming diagonal lines (indicating predictability) |
| **L** – Average Diagonal Line Length | Mean time over which the system exhibits similar behavior |
| **ENTR** – Entropy | Shannon entropy of diagonal line lengths (complexity measure) |
| **LAM** – Laminarity | Fraction of points forming vertical lines (indicating intermittency or stationarity) |
| **TT** – Trapping Time | Average vertical line length |

These metrics help detect patterns such as synchronization, chaos, and transitions in dynamics.

---

#### 2. Kuramoto Model & Synchronization

The **Kuramoto model** is a classical model for understanding synchronization in populations of coupled oscillators (e.g., neurons, fireflies, mechanical rotors).

Each oscillator $i$ has a phase $\theta_i(t)$, and their interaction is governed by:

\begin{equation}
    \frac{d\theta_i}{dt} = \omega_i + \frac{K}{N} \sum_{j=1}^{N} \sin(\theta_j - \theta_i),
\end{equation}

where:
- $\omega_i$ is the natural frequency,
- $K$ is the coupling strength,
- $N$ is the number of oscillators.

The **degree of synchronization** is measured by the **Kuramoto order parameter** $R(t)$:

\begin{equation}
    R(t) = \left| \frac{1}{N} \sum_{j=1}^{N} e^{i \theta_j(t)} \right|
\end{equation}

- $R(t) \in [0, 1]$
- $R(t) \approx 1$: perfect phase synchronization
- $R(t) \approx 0$: desynchronized or incoherent state

This metric summarizes global coherence in the system and is often visualized as a function of time or coupling strength $K$.

The **Kuramoto parameter** offers a global phase measure, while **RQA** captures finer nonlinear structures and local synchrony.

---

In [17]:
Rs, df_metrics = esa.estimate_multiple_crqa(
    df=df_final,
    epsilon_factor=0.05,
    plot=False,
    min_diagonal_length=2,
    min_vertical_length=2
)
df_metrics.to_csv(output_path + "/df_recurrence_sim.csv", index=False)
df_metrics

,video,id_pair,RR,DET,L,ENTR,LAM,TT,num_diag_lines,num_vert_lines
0,2n_2m_0f_simulation,00,0.031726,0.345556,2.273616,0.325677,0.732676,2.134726,6955,15706
1,2n_2m_0f_simulation,01,0.016328,0.366736,2.368248,0.134849,0.782727,2.068215,3647,8913
2,2n_2m_0f_simulation,11,0.439424,0.562901,2.768813,1.089496,0.791993,3.859731,128857,130057
3,3n_3m_0f_simulation,00,0.029805,0.305762,2.281722,0.266328,0.699100,2.086863,5761,14402
4,3n_3m_0f_simulation,01,0.011831,0.356226,2.502676,0.054012,0.756754,2.019390,2429,6395
5,3n_3m_0f_simulation,02,0.002262,0.388906,36.257143,0.129741,0.304321,2.014199,35,493
6,3n_3m_0f_simulation,11,0.326000,0.523622,2.410721,0.817912,0.794385,3.049663,102135,122485
7,3n_3m_0f_simulation,12,0.011526,0.151038,3.869029,0.115578,0.342797,2.037540,649,2797
8,3n_3m_0f_simulation,22,0.029757,0.260222,2.322520,0.267820,0.620629,2.099133,4809,12690
9,4n_4m_0f_simulation,00,0.029585,0.311227,2.288644,0.288025,0.702411,2.105951,5803,14233


In [18]:
# Estimate Kuramoto order parameter
df_synchronization = []
for video in df_final["video"].unique():
    mask = df_final["video"] == video
    times, Rs = esa.estimate_kuramoto_order_parameter(df=df_final[mask])
    df_synchronization.append(pd.DataFrame({
        "video": [video]*len(times),
        "time": times,
        "order_parameter": Rs
    }))
df_synchronization = pd.concat(df_synchronization, ignore_index=True)
df_synchronization.to_csv(output_path + "/df_kuramoto_sim.csv", index=False)

# How different is the behavior with different number of cockroaches?

- Aim: To quantify network properties (modularity, clustering, degree distribution).
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.

## Get Visibility Graph between individuals
The **Visibility Graph** ($VG$) is a method that transforms time series into a complex network, where data points in the series are mapped to nodes and edges are created based on visibility criteria.

### **Mathematical Formulation**
Given a time series $X = \{ x_1, x_2, ..., x_N \}$, two data points $(x_i, t_i)$ and $(x_j, t_j)$ are connected if any intermediate point $(x_k, t_k)$ satisfies:

\begin{equation}
    x_k < x_i + (x_j - x_i) \frac{t_k - t_i}{t_j - t_i}, \quad \forall k \in (i,j)
\end{equation}

In [19]:
window_sizes = [
    [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225], [240],
    [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_network_all, df_network_nodes = [], []
for video in df_final["video"].unique():
    df = df_final[df_final["video"] == video]
    df_all, df_nodes = ecna.estimate_multiple_vg(
        df=df,
        window_sizes=window_sizes,
        log_path=log_path,
        log_filename="log_network_sim",
        verbose=1,
        tqdm_bar=True
    )
    df_network_all.append(df_all)
    df_network_nodes.append(df_nodes)

df_network_all = pd.concat(df_network_all, ignore_index=True)
df_network_all.to_csv(output_path + "/df_network_sim.csv", index=False)

df_network_nodes = pd.concat(df_network_nodes, ignore_index=True)
df_network_nodes.to_csv(output_path + "/df_network_nodes_sim.csv", index=False)

100%|███████████████████████| 23/23 [01:17<00:00,  3.38s/it]
